In [309]:
import opt_einsum as oe
import numpy as np
import torch
import sys
sys.path.append("../../")
import mps
from mps.trainer.data_utils import SyntheticDataset, SyntheticDatasetV2

In [310]:
N = 256
dataset = SyntheticDatasetV2(n=N, num_samples=(2**15), seed=42)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2**10, shuffle=True)

In [214]:
from mps.trainer.smps_trainer import smps_train
from mps.simple_mps import SimpleMPS
import copy
chi = 2
d = 2
l = 2
device = torch.device("cpu")
dtype = torch.float64
optimize = "greedy"
eps = 1e-2
smps = SimpleMPS(N, chi, d, l, layers=1, device=device, dtype=dtype, optimize=optimize, eps=eps)

Path is not set, setting...
Found the path
Initialized MPS with random matrices


In [215]:
from mps.trainer.smps_trainer import smps_train
import copy

data, target = next(iter(dataloader))

smps.mps.params[0] = torch.eye(2).to(torch.float64)
for i in range(1, len(smps.mps.params) - 1):
    A = torch.stack([torch.eye(2), torch.eye(2).flip(1)]).to(torch.float64)
    A.permute(1, 0, 2)
    smps.mps.params[i] = A

smps.mps.params[-1] = torch.eye(2).to(torch.float64)

smps.initialize_MPS()

(smps(data.permute(1, 0, 2)).argmax(dim=-1) - target).sum()


smps = smps_train(
    dataloader,
    N=N,
    d=2,
    l=2,
    chi=2,
    eps=1e-2,
    epochs=1,
    lr=0.1,
    log_steps=10,
    dtype=torch.float64,
    device=torch.device("cpu"),
    optimize="greedy",
    optimizer=torch.optim.Adam,
    smps=copy.deepcopy(smps),
)

Initialized MPS with random matrices

=== Training SimpleMPS for 1 epoch(s)... ===
[SimpleMPS] Epoch 1, Step 1/32 | Loss: 0.313262 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 2/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 3/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 4/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 5/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 6/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 7/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 8/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 9/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 10/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 11/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 12/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 13/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 14/32 | Loss: 0.000000 | Acc: 100.00%
[SimpleMPS] Epoch 1, Step 15/3

In [321]:
from mps import tpcp_mps
from mps.trainer.utils import calculate_accuracy, loss_batch
# --- Step 2: Build and Prepare TPCP ---
tpcp = tpcp_mps.MPSTPCP(
    N,
    K=8,
    d=2,
    enable_r=False,
    with_identity=False,
    manifold=tpcp_mps.ManifoldType.EXACT,
)
tpcp.train()
tpcp.set_canonical_mps(smps)

logsoftmax = torch.nn.LogSoftmax(dim=-1)
nnloss = torch.nn.NLLLoss(reduction="mean")

# Initialize W: start with first column ones and second column small.

Kraus operator is not on the Stiefel manifold: `X^T X != I` with atol=1e-05, rtol=1e-05


In [361]:
W = torch.zeros(tpcp.L, 2, dtype=torch.float64)
W[:, 0] = 1 
W[:, 1] = 1
tpcp.initialize_W(W)

# --- Step 3: Determine lambda_final Using the Initial Loss Value ---
data_batch, target_batch = next(iter(dataloader))
initial_probs, reg = tpcp(data_batch, return_probs=True, return_reg=True)
# softmax_initial_probs = logsoftmax(initial_probs)
initial_accuracy = calculate_accuracy(initial_probs[:, 0], target_batch)
print(f"Initial accuracy: {initial_accuracy.item():.2%}")

loss = loss_batch(initial_probs[:, 0], target_batch)
print(f"Initial loss: {loss.item()}")

Initial accuracy: 100.00%
Initial loss: 0.024106526865987313


In [362]:
# from mps.trainer.adaptive_mpsae_trainer import RiemannianAdam
from mps.radam import RiemannianAdam
import torch
lr = 0.001
optimizer = RiemannianAdam(tpcp.kraus_ops.parameters(), lr=lr, betas=(0.99, 0.999))
optimizer_weight = torch.optim.Adam([tpcp.r], lr=lr)
tpcp.W.requires_grad = False

In [363]:
clambda = 0
epochs = 1000
for _ in range(epochs):
    for data, target in dataloader:
        optimizer.zero_grad()
        optimizer_weight.zero_grad()
        outputs, reg = tpcp(data, return_probs=True, return_reg=True)
        # probs = logsoftmax(outputs)
        # loss0 = nnloss(probs, target)
        loss0 = loss_batch(outputs[:, 0], target)
        loss = loss0  + clambda * reg

        loss.backward()

        optimizer.step()
        optimizer_weight.step()

        tpcp.proj_stiefel(check_on_manifold=True, print_log=False, rtol=1e-3)
        tpcp.normalize_w_and_r()

        acc = calculate_accuracy(outputs[:, 0], target)
        srpq = torch.exp(-reg)
        print(f"Loss0 : {loss0.item():.6f}, reg: {reg.item():.6f}, Loss: {loss.item():.6f}, Acc: {acc.item():.2%}, SRPQ: {srpq.item():.6e}")


Loss0 : 0.024195, reg: 0.000000, Loss: 0.024195, Acc: 100.00%, SRPQ: 1.000000e+00
Loss0 : 0.023142, reg: 0.000000, Loss: 0.023142, Acc: 100.00%, SRPQ: 1.000000e+00
Loss0 : 0.014204, reg: 0.000000, Loss: 0.014204, Acc: 100.00%, SRPQ: 1.000000e+00
Loss0 : 0.011504, reg: 0.000000, Loss: 0.011504, Acc: 100.00%, SRPQ: 1.000000e+00
Loss0 : 0.012087, reg: 0.000000, Loss: 0.012087, Acc: 100.00%, SRPQ: 1.000000e+00
Loss0 : 0.011595, reg: 0.000000, Loss: 0.011595, Acc: 100.00%, SRPQ: 1.000000e+00
Loss0 : 0.010191, reg: 0.000000, Loss: 0.010191, Acc: 100.00%, SRPQ: 1.000000e+00
Loss0 : 0.009235, reg: 0.000000, Loss: 0.009235, Acc: 100.00%, SRPQ: 1.000000e+00


KeyboardInterrupt: 

In [394]:
i = 1
x = dataset[i][0]

rho0 = torch.einsum("i, j->ij", x[0], x[0].conj())
rho1 = torch.einsum("i, j->ij", x[1], x[1].conj())
rho_in = torch.kron(rho0, rho1)

K = tpcp.kraus_ops[0].data.reshape(8, 4, 4)
rho_k_out = torch.einsum("kij, jl, kml->im", K, rho_in, K.conj()).reshape(2, 2, 2, 2)
# print(rho_k_out.reshape(4, 4))
#partial trace
rho_k_out = torch.einsum("ijil -> jl", rho_k_out)

for k in range(1, 3,):
    K = tpcp.kraus_ops[k].data.reshape(8, 4, 4)
    rho_new = torch.einsum("i, j->ij", x[k+1], x[k+1].conj())
    rho_k_in = torch.kron(rho_k_out, rho_new)
    rho_k_out = torch.einsum("kij, jl, kml->im", K, rho_in, K.conj()).reshape(2, 2, 2, 2)
    rho_last = rho_k_out.data.reshape(4, 4)
    #partial trace
    rho_k_out = torch.einsum("ijil -> jl", rho_k_out)

tensor([[ 0.0838,  0.2242, -0.0170, -0.0440],
        [ 0.2242,  0.8752, -0.0452, -0.1149],
        [-0.0170, -0.0452,  0.0053,  0.0138],
        [-0.0440, -0.1149,  0.0138,  0.0357]], dtype=torch.float64)
tensor([[ 0.0636, -0.1798,  0.0134, -0.0390],
        [-0.1798,  0.8929, -0.0391,  0.1087],
        [ 0.0134, -0.0391,  0.0045, -0.0133],
        [-0.0390,  0.1087, -0.0133,  0.0389]], dtype=torch.float64)


In [391]:
mes0 = tpcp.r.conj() @ tpcp.pros0 @ tpcp.r.T - tpcp.r.conj() @ tpcp.pros1 @ tpcp.r.T

torch.trace(mes0 @ rho_k_out), dataset[i][1]

(tensor(-0.3303, dtype=torch.float64, grad_fn=<TraceBackward0>), tensor(1))

In [393]:
tpcp.rho_list[1]

tensor([[[ 0.0000,  0.0000, -0.0000, -0.0000],
         [ 0.0000,  0.0325, -0.0000, -0.1773],
         [-0.0000, -0.0000,  0.0000,  0.0000],
         [-0.0000, -0.1773,  0.0000,  0.9675]]], dtype=torch.float64,
       grad_fn=<ViewBackward0>)

In [370]:
tpcp(data_batch[i].unsqueeze(0)), target_batch[i]

(tensor([0.0095], dtype=torch.float64, grad_fn=<SelectBackward0>), tensor(1))

In [371]:
tpcp.rho_list[0]

tensor([[[ 0.0308, -0.1679,  0.0054, -0.0291],
         [-0.1679,  0.9157, -0.0295,  0.1593],
         [ 0.0054, -0.0295,  0.0017, -0.0095],
         [-0.0291,  0.1593, -0.0095,  0.0518]]], dtype=torch.float64,
       grad_fn=<SumBackward1>)